# ChartNarrator Stage 2 VLM fine-tuning (WA >= 4.5 matched)

Public cleanup of the original `train_0406_vlm_4_5` comparison notebook. The training and inference logic is preserved, while local absolute paths, Chinese comments/output text, stale sample-count comments, private Colab paths, and long execution outputs have been removed.

This notebook corresponds to the matched VLM 4.5 condition used for controlled comparison with the VLM 5.0 condition. It uses the WA >= 4.5 training pool, but with the matched dataset size and held-out test set used in the comparison setup.

Original run record from the cleaned notebook outputs:

- Model: `Qwen/Qwen2-VL-7B-Instruct`
- Dataset split: train = 705, validation = 88, test = 88
- Fine-tuning: QLoRA, 4-bit NF4, LoRA rank = 64, alpha = 32, epochs = 8
- Total optimization steps: 352
- Final train loss: 0.8718
- Final eval loss: 0.0936
- Test inference: 88 / 88 successful
- Format compliance: P1 = 100%, P2 = 100%, P3 = 100%, full format compliance = 100.0%


## 1. Configure repository paths

This cell locates the repository root, defines the matched VLM 4.5 dataset, checkpoint, and prediction-output paths, and prints split counts when the data is already present.


In [ ]:
import json
import os
from pathlib import Path

# Set CHARTNARRATOR_ROOT when running outside the repository root.
# Example: os.environ["CHARTNARRATOR_ROOT"] = "/content/ChartNarrator_public"
def resolve_project_root():
    env_root = os.environ.get("CHARTNARRATOR_ROOT")
    if env_root:
        return Path(env_root).resolve()
    cwd = Path.cwd().resolve()
    for candidate in [cwd] + list(cwd.parents):
        if (candidate / "data").exists() or (candidate / "scripts").exists():
            return candidate
    return cwd

PROJECT_ROOT = resolve_project_root()

DATA_DIR = PROJECT_ROOT / "data" / "finetune_dataset_4_5_matched"
IMAGE_DIR = DATA_DIR / "images"
CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints" / "qwen2_vl_chartnarrator_stage2_4_5_matched"
PREDICTION_OUTPUT = PROJECT_ROOT / "data" / "evaluation" / "predictions" / "predictions_test_stage2_4_5_matched_round3.json"

print(f"Project root : {PROJECT_ROOT}")
print(f"Dataset dir  : {DATA_DIR}")
print(f"Image dir    : {IMAGE_DIR}")
print(f"Checkpoint   : {CHECKPOINT_DIR}")
print(f"Predictions  : {PREDICTION_OUTPUT}")

for split in ["train", "val", "test"]:
    split_path = DATA_DIR / f"{split}.json"
    if split_path.exists():
        with open(split_path, encoding="utf-8") as f:
            print(f"{split}.json: {len(json.load(f))} entries")
    else:
        print(f"{split}.json: missing")
print(f"images: {len(list(IMAGE_DIR.glob('*.png'))) if IMAGE_DIR.exists() else 0} PNG files")


## 2. Inspect one matched training example

This optional sanity check shows the first user prompt, target response, and image path. It also confirms that `[CHART CONTEXT]` is present in the matched VLM condition.


In [ ]:
import json

with open(DATA_DIR / "train.json", encoding="utf-8") as f:
    train = json.load(f)

e = train[0]
user_prompt = e["conversations"][0]["value"]

print("=== user prompt preview (first 300 chars) ===")
print(user_prompt[:300])
print()
print("Contains [CHART CONTEXT]:", "[CHART CONTEXT]" in user_prompt)
print()
print("=== assistant target preview (first 200 chars) ===")
print(e["conversations"][1]["value"][:200])
print()
print("=== image field ===")
print(e.get("image", "NO IMAGE FIELD"))


## 3. Normalize image paths for LLaMA-Factory

This cell preserves the original preprocessing step: each split JSON receives absolute image paths, role tags are checked, and `<image>` is inserted if the user turn is missing it.


In [ ]:
import json
import os

TARGET_FILES = ["train.json", "val.json", "test.json"]

for filename in TARGET_FILES:
    file_path = DATA_DIR / filename
    if not file_path.exists():
        print(f"Skipping missing split: {filename}")
        continue

    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    fixed_count = 0
    missing_images = 0
    role_issues = 0

    for entry in data:
        img_rel = entry.get("image", "")
        fname = os.path.basename(img_rel.replace("\\", "/"))
        img_abs = IMAGE_DIR / fname
        entry["image"] = str(img_abs)

        if not img_abs.exists():
            missing_images += 1

        convs = entry.get("conversations", [])
        if len(convs) >= 2:
            if convs[0]["from"] not in ["user", "human"]:
                role_issues += 1
            if convs[1]["from"] not in ["assistant", "gpt"]:
                role_issues += 1

            first = convs[0]
            if first["from"] in ["user", "human"] and "<image>" not in first["value"]:
                first["value"] = "<image>" + first["value"]
                fixed_count += 1

    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

    status = "OK" if missing_images == 0 else "WARN"
    print(
        f"[{status}] {filename}: {len(data)} entries | "
        f"added <image>: {fixed_count} | missing images: {missing_images} | role issues: {role_issues}"
    )

print("Dataset preprocessing complete.")


## 4. Verify or install the training environment

This cell checks the expected PyTorch, Transformers, Accelerate, and LLaMA-Factory versions. Installation is only triggered when the stack is missing or incompatible.


In [ ]:
import importlib
import os
import sys


def check_versions():
    """Return True when the required training stack is already installed."""
    try:
        import torch
        import transformers
        import accelerate
        import llamafactory
        ok = (
            torch.__version__.startswith("2.5.1")
            and transformers.__version__ == "4.46.1"
            and accelerate.__version__ == "1.0.1"
        )
        if ok:
            print("Training environment is ready; installation skipped.")
            print(
                f"PyTorch {torch.__version__} | "
                f"Transformers {transformers.__version__} | "
                f"Accelerate {accelerate.__version__}"
            )
            return True
    except ImportError:
        pass
    return False


if check_versions():
    pass
else:
    print("Removing incompatible packages...")
    os.system("pip uninstall -y torch torchvision torchaudio accelerate transformer-engine flash-attn")

    print("Installing PyTorch 2.5.1...")
    os.system(
        "pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 "
        "--index-url https://download.pytorch.org/whl/cu121"
    )

    print("Installing training dependencies...")
    os.system("pip install transformers==4.46.1 accelerate==1.0.1 peft==0.12.0 bitsandbytes==0.44.1")
    os.system("pip install datasets==2.21.0 trl==0.9.6 scipy einops sentencepiece protobuf tiktoken")
    os.system("pip install deepspeed==0.15.4")
    os.system("pip install flash-attn==2.7.4.post1 --no-build-isolation")

    print("Installing LLaMA-Factory v0.9.1...")
    os.system("git clone --depth 1 -b v0.9.1 https://github.com/hiyouga/LLaMA-Factory.git")
    os.chdir("LLaMA-Factory")
    os.system("pip install -e .[torch,metrics]")
    os.chdir("..")

    print("Installation finished. Restart the runtime, then continue from the next cell.")


## 5. Write `dataset_info.json`

This cell registers the matched train, validation, and test split names in the LLaMA-Factory ShareGPT format.


In [ ]:
import json

INFO_PATH = DATA_DIR / "dataset_info.json"

dataset_info = {}
for split in ["train", "val", "test"]:
    dataset_info[f"stage2_4_5_matched_{split}"] = {
        "file_name": f"{split}.json",
        "formatting": "sharegpt",
        "columns": {
            "messages": "conversations",
            "images": "image",
        },
        "tags": {
            "role_tag": "from",
            "content_tag": "value",
            "user_tag": "user",
            "assistant_tag": "assistant",
        },
    }

with open(INFO_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset_info, f, indent=2, ensure_ascii=False)

print(f"dataset_info.json written to: {INFO_PATH}")
for key in dataset_info:
    print(f"dataset key: {key}")


## 6. Write the QLoRA training configuration

This cell creates the LLaMA-Factory YAML config. The model, 4-bit quantization, LoRA parameters, learning rate, batch size, and epoch count reproduce the original matched VLM 4.5 run.


In [ ]:
import json
import os
import yaml

split_counts = {}
for split in ["train", "val", "test"]:
    split_path = DATA_DIR / f"{split}.json"
    if split_path.exists():
        split_counts[split] = len(json.loads(split_path.read_text(encoding="utf-8")))
    else:
        split_counts[split] = "missing"

train_args = {
    "model_name_or_path": "Qwen/Qwen2-VL-7B-Instruct",
    "stage": "sft",
    "do_train": True,
    "finetuning_type": "lora",
    "quantization_bit": 4,
    "template": "qwen2_vl",
    "flash_attn": "fa2",

    "dataset_dir": str(DATA_DIR),
    "dataset": "stage2_4_5_matched_train",
    "eval_dataset": "stage2_4_5_matched_val",

    "cutoff_len": 4096,
    "learning_rate": 1e-5,
    "num_train_epochs": 8.0,

    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,

    "eval_strategy": "steps",
    "eval_steps": 100,
    "save_steps": 200,
    "logging_steps": 5,
    "save_total_limit": 3,

    "lora_rank": 64,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target": "all",

    "output_dir": str(CHECKPOINT_DIR),
    "overwrite_output_dir": True,
    "plot_loss": True,
    "bf16": True,
    "fp16": False,
    "ddp_timeout": 180000000,
}

config_path = PROJECT_ROOT / "train_config_stage2_4_5_matched.yaml"
with open(config_path, "w") as f:
    yaml.dump(train_args, f)

print(f"Training config written to: {config_path}")
print(
    "Dataset: "
    f"train={split_counts['train']} | val={split_counts['val']} | test={split_counts['test']}"
)
print("LoRA: rank=64 | alpha=32 | epochs=8")
print("Input: image + anchor text ([CHART CONTEXT] injected into user prompt)")
print(f"Output checkpoint dir: {CHECKPOINT_DIR}")


## 7. Launch fine-tuning

This long-running cell starts QLoRA fine-tuning from the generated YAML config. Run it only in a GPU environment with the required model and dataset files available.


In [ ]:
import torch

torch.cuda.empty_cache()

print("Stage 2 VLM matched training start")
print("Expected comparison split: train=705 | val=88 | test=88")
print("LoRA rank=64 | alpha=32 | epochs=8")
!llamafactory-cli train {str(config_path)}


## 8. Run matched test-set inference

This cell loads the 4-bit base model, attaches the matched VLM 4.5 LoRA adapter, keeps the checkpoint processor, removes the `<image>` placeholder from the text prompt, and writes test predictions.


In [ ]:
!pip install qwen-vl-utils -q

import json
import os

import torch
from peft import PeftModel
from qwen_vl_utils import process_vision_info
from tqdm import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig, GenerationConfig, Qwen2VLForConditionalGeneration

TEST_FILE = DATA_DIR / "test.json"
PREDICTION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("ChartNarrator Stage 2 VLM matched inference")
print("=" * 70)
print(f"Checkpoint : {CHECKPOINT_DIR}")
print(f"Test set   : {TEST_FILE}")
print(f"Output     : {PREDICTION_OUTPUT}")
print("=" * 70)

print("Loading base model in 4-bit NF4, matching the training quantization setup...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-7B-Instruct",
    quantization_config=bnb_config,
    device_map="cuda",
)

model = PeftModel.from_pretrained(base_model, str(CHECKPOINT_DIR))
model.eval()

model.generation_config = GenerationConfig(
    bos_token_id=151643,
    eos_token_id=151645,
    pad_token_id=151643,
)

processor = AutoProcessor.from_pretrained(str(CHECKPOINT_DIR))
print("Model loaded.")

test_msg = [{"role": "user", "content": [{"type": "text", "text": "test"}]}]
test_text = processor.apply_chat_template(test_msg, tokenize=False, add_generation_prompt=True)
print("Template preview (first 200 chars):")
print(repr(test_text[:200]))

with open(TEST_FILE, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print(f"Test samples: {len(test_data)}")

predictions = []

for idx, entry in enumerate(tqdm(test_data, desc="Inference")):
    try:
        img_abs = entry["image"]
        user_query = entry["conversations"][0]["value"].replace("<image>", "").strip()

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img_abs},
                    {"type": "text", "text": user_query},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_new_tokens=2048,
                do_sample=False,
                repetition_penalty=1.1,
            )

        generated_ids_trimmed = [
            out_ids[len(in_ids):]
            for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )[0]

        has_p1 = "── Paragraph 1" in output_text
        has_p2 = "── Paragraph 2" in output_text
        has_p3 = "── Paragraph 3" in output_text

        predictions.append(
            {
                "id": entry["id"],
                "image": entry["image"],
                "morphology_family": entry.get("morphology_family", ""),
                "route_label": entry.get("route_label", ""),
                "conversations": [
                    {"from": "user", "value": user_query},
                    {"from": "assistant", "value": output_text},
                ],
                "reference": entry["conversations"][1]["value"],
                "format_check": {"p1": has_p1, "p2": has_p2, "p3": has_p3},
            }
        )

        if idx == 0:
            print("\n" + "=" * 70)
            print(f"First sample preview ({entry['id']})")
            print("=" * 70)
            print(output_text[:600])
            print("=" * 70 + "\n")

    except Exception as e:
        print(f"Sample {idx} ({entry.get('id', '?')}) failed: {e}")
        predictions.append({"id": entry.get("id", ""), "error": str(e)})

with open(PREDICTION_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

successful = [p for p in predictions if "error" not in p]
print("\n" + "=" * 70)
print("Inference statistics")
print("=" * 70)
print(f"Success: {len(successful)} / {len(predictions)}")
if successful:
    p1 = sum(1 for p in successful if p["format_check"]["p1"]) / len(successful) * 100
    p2 = sum(1 for p in successful if p["format_check"]["p2"]) / len(successful) * 100
    p3 = sum(1 for p in successful if p["format_check"]["p3"]) / len(successful) * 100
    all_fmt = sum(1 for p in successful if all(p["format_check"].values())) / len(successful) * 100
    print(f"Format: P1={p1:.0f}% | P2={p2:.0f}% | P3={p3:.0f}%")
    print(f"Full format compliance: {all_fmt:.1f}%")
print(f"Results saved to: {PREDICTION_OUTPUT}")


## 9. Optional checkpoint sanity check

This optional verification cell lists checkpoint files and checks whether `adapter_config.json` exists. It is useful before running inference but is not part of the experimental method itself.


In [ ]:
import os

print("Files in checkpoint dir:")
if CHECKPOINT_DIR.exists():
    for file_name in os.listdir(CHECKPOINT_DIR):
        print(f" {file_name}")
    print("\nadapter_config.json exists:", (CHECKPOINT_DIR / "adapter_config.json").exists())
else:
    print(f"Checkpoint directory not found: {CHECKPOINT_DIR}")
